# PhishNet-Transformer — Step 2: Classic ML Baseline (XGBoost)

**Goal of this notebook:** build the classic machine learning half of the project — hand-engineered lexical/host-based features from the raw URL, fed into an XGBoost classifier — trained and evaluated on the *exact same* `train.csv` / `test.csv` split from Step 1.

**Why this matters:** this becomes your baseline number. Everything DistilBERT does later gets compared against this. If we used a different split or different data for the two models, the comparison would be meaningless — so this step deliberately reuses Step 1's files unchanged.

**What you need before running this:** `train.csv`, `val.csv`, and `test.csv` from Step 1, in the same folder as this notebook.

## Cell 1 — Load the splits from Step 1

**Why:** we're not touching the original PhiUSIIL file again — everything from here on builds only on the clean, balanced, split data we already produced and saved in Step 1.

In [1]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)
train.head()

Train: (8400, 2) Val: (1800, 2) Test: (1800, 2)


,URL,label
0,https://www.plasticfreejuly.org,0
1,https://provi78arge.webcindario.com/,1
2,https://www.roseyleebooks.com,0
3,https://att-106905.weeblysite.com/,1
4,https://tinyurl.com/blocca-pagamento,1


## Cell 2 — Define the feature extraction function

**Why:** XGBoost (like most classic ML models) can't read raw text directly — it needs numeric input. So we hand-engineer a set of numeric signals from each URL string that are known, interpretable indicators of phishing. This is exactly the kind of feature engineering real lexical/host-based phishing detectors use.

**The 12 features, and why each one is a real phishing signal:**
- `url_length` — phishing URLs are often unusually long (padded to hide the real domain)
- `num_dots` — excessive subdomains (e.g. `secure.login.bank.fake-site.com`) are a common obfuscation trick
- `num_hyphens` — attackers often insert hyphens to mimic brand names (`paypal-secure-login.com`)
- `num_digits` — randomly generated or auto-created malicious domains often contain more digits
- `num_special_chars` — unusual characters can indicate obfuscation attempts
- `has_https` — legitimate sites are more consistently HTTPS (though phishing sites increasingly use HTTPS too — the model will learn how much this feature actually matters from the data itself)
- `has_ip` — using a raw IP address instead of a domain name is a classic phishing red flag
- `num_subdirs` — deep, unusual path structures can indicate disguised malicious pages
- `domain_length` — very short or very long domains are both slightly unusual
- `has_at_symbol` — the `@` symbol in a URL can be used to trick browsers about the real destination
- `suspicious_word_count` — counts words like "login", "verify", "secure", "account", "bank" — common social-engineering bait words
- `entropy` — a measure of how "random-looking" the characters are; auto-generated malicious domains tend to have higher entropy than natural words

In [2]:
def extract_features(url):
    url = str(url)
    parsed = urlparse(url if '://' in url else 'http://' + url)
    domain = parsed.netloc

    length = len(url)
    num_dots = url.count('.')
    num_hyphens = url.count('-')
    num_digits = sum(c.isdigit() for c in url)
    num_special = len(re.findall(r'[^a-zA-Z0-9.\-/:]', url))
    has_https = int(parsed.scheme == 'https')
    has_ip = int(bool(re.match(r'^(\d{1,3}\.){3}\d{1,3}$', domain)))
    num_subdirs = url.count('/')
    domain_length = len(domain)
    at_symbol = int('@' in url)
    suspicious_words = ['login','verify','update','secure','account','bank','confirm','signin','webscr']
    suspicious_count = sum(w in url.lower() for w in suspicious_words)

    # Shannon entropy: measures how "random" the character distribution is
    probs = [url.count(c)/length for c in set(url)] if length > 0 else [0]
    entropy = -sum(p*np.log2(p) for p in probs if p > 0)

    return pd.Series({
        'url_length': length, 'num_dots': num_dots, 'num_hyphens': num_hyphens,
        'num_digits': num_digits, 'num_special_chars': num_special, 'has_https': has_https,
        'has_ip': has_ip, 'num_subdirs': num_subdirs, 'domain_length': domain_length,
        'has_at_symbol': at_symbol, 'suspicious_word_count': suspicious_count, 'entropy': entropy
    })

# quick test on one example URL
extract_features("https://provi78arge.webcindario.com/")

url_length               36.000000
num_dots                  2.000000
num_hyphens               0.000000
num_digits                2.000000
num_special_chars         0.000000
has_https                 1.000000
has_ip                    0.000000
num_subdirs               3.000000
domain_length            27.000000
has_at_symbol             0.000000
suspicious_word_count     0.000000
entropy                   4.308271
dtype: float64

**Interpretation of the test call:** you should see 12 numeric values printed for that one phishing example — check that `has_ip` is 0 (it's a domain, not an IP), `has_https` is 1, and `entropy` is a number roughly between 3 and 5 for most real-world URLs. If any value looks like `NaN` or throws an error, there's likely an unusual URL format we need to handle — flag it before moving on.

## Cell 3 — Apply feature extraction to all three splits

**Why:** now we turn every URL in train, validation, and test into its 12-number feature representation. `.apply()` runs our function on every row.

In [3]:
feat_train = train['URL'].apply(extract_features)
feat_val = val['URL'].apply(extract_features)
feat_test = test['URL'].apply(extract_features)

print("Feature matrix shape (train):", feat_train.shape)
feat_train.describe()

Feature matrix shape (train): (8400, 12)


,url_length,num_dots,num_hyphens,num_digits,num_special_chars,has_https,has_ip,num_subdirs,domain_length,has_at_symbol,suspicious_word_count,entropy
count,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.00000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000
mean,37.295119,2.264643,0.418690,2.383929,0.324405,0.743929,0.00250,2.515595,21.907619,0.007024,0.041667,3.977515
std,58.368660,1.340242,2.908504,23.348203,4.920526,0.436488,0.04994,1.377275,9.654764,0.083518,0.235906,0.327114
min,15.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,2.000000,4.000000,0.000000,0.000000,2.623429
25%,25.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.00000,2.000000,16.000000,0.000000,0.000000,3.772055
50%,29.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.00000,2.000000,20.000000,0.000000,0.000000,3.940555
75%,36.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.00000,3.000000,25.000000,0.000000,0.000000,4.116300
max,4247.000000,90.000000,250.000000,2011.000000,390.000000,1.000000,1.00000,68.000000,105.000000,1.000000,3.000000,5.335261


**Interpretation:** `.describe()` shows the distribution of each feature. Two things worth checking: (1) does `url_length` have a sensible range (a handful of very long URLs pulling the max up is normal and fine), and (2) is `suspicious_word_count` mostly 0 with some higher values (expected — most URLs won't contain bait words, only some phishing ones will)? If a column is entirely 0 or entirely one value, it won't help the model at all — worth noting, but not necessarily a problem.

## Cell 4 — Train the XGBoost model

**Why XGBoost specifically:** it's the standard, strong-performing choice for structured/tabular feature data like this (same reasoning as your churn project), and it handles the mix of feature scales here (counts, ratios, binary flags) without needing manual scaling.

**What we're doing:** fitting the model on the training features and labels only. The validation set is intentionally not used here — for XGBoost with these settings we don't need it, but it stays available for later if you want to tune hyperparameters.

In [4]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)

model.fit(feat_train, train['label'])
print("Model trained.")

Model trained.


## Cell 5 — Evaluate on the test set

**Why the test set specifically:** this is data the model has never seen in any form — not during training, not during any tuning. This is the only honest way to measure how the model would perform on genuinely new URLs.

**The four metrics, and what each one tells you:**
- **Accuracy** — percent of all predictions that were correct. Easy to understand, but can be misleading if classes are imbalanced (not a concern here since we balanced the data in Step 1).
- **Precision** — of the URLs the model flagged as phishing, what percent actually were? Low precision means too many false alarms (legitimate sites wrongly blocked).
- **Recall** — of all the actual phishing URLs, what percent did the model catch? Low recall means real phishing sites are slipping through.
- **F1** — the balance between precision and recall in one number; useful for a single headline metric.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

pred_test = model.predict(feat_test)

acc = accuracy_score(test['label'], pred_test)
prec = precision_score(test['label'], pred_test)
rec = recall_score(test['label'], pred_test)
f1 = f1_score(test['label'], pred_test)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 score:  {f1:.4f}")

print("\nFull classification report:")
print(classification_report(test['label'], pred_test, target_names=['legitimate (0)', 'phishing (1)']))

Accuracy:  0.9928
Precision: 0.9966
Recall:    0.9889
F1 score:  0.9927

Full classification report:
                precision    recall  f1-score   support

legitimate (0)       0.99      1.00      0.99       900
  phishing (1)       1.00      0.99      0.99       900

      accuracy                           0.99      1800
     macro avg       0.99      0.99      0.99      1800
  weighted avg       0.99      0.99      0.99      1800



## Cell 6 — Confusion matrix

**Why:** the four metrics above are summaries — the confusion matrix shows exactly *where* the model's mistakes are, which is far more useful when you're explaining the model's behavior in an interview.

In [6]:
cm = confusion_matrix(test['label'], pred_test)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: legitimate', 'Actual: phishing'],
    columns=['Predicted: legitimate', 'Predicted: phishing']
)
cm_df

,Predicted: legitimate,Predicted: phishing
Actual: legitimate,897,3
Actual: phishing,10,890


**How to read this table:** the top-left and bottom-right numbers are correct predictions (true negatives and true positives). The top-right number is legitimate URLs wrongly flagged as phishing (false positives — annoying but not dangerous). The bottom-left number is phishing URLs the model missed (false negatives — the more dangerous kind of mistake, since a real attack gets through). Compare these two error types — whichever is larger tells you where the model's weakness actually is.

## Cell 7 — Feature importance

**Why:** this shows which of your 12 hand-engineered features the model actually relied on most — useful both as a sanity check (do the important features make intuitive sense?) and as material for your interview explanation.

In [7]:
importances = pd.Series(model.feature_importances_, index=feat_train.columns).sort_values(ascending=False)
importances

has_https                0.495668
num_subdirs              0.486802
num_dots                 0.005370
url_length               0.004893
num_digits               0.003450
num_hyphens              0.002402
entropy                  0.000852
domain_length            0.000563
has_ip                   0.000000
num_special_chars        0.000000
has_at_symbol            0.000000
suspicious_word_count    0.000000
dtype: float32

**Interpretation:** whichever features sit at the top are what the model leans on most heavily to distinguish phishing from legitimate URLs. If `url_length`, `entropy`, or `num_dots` rank highly, that lines up with the well-known real-world pattern that phishing URLs tend to be longer, more "random-looking," and more heavily subdomained than legitimate ones — a good, explainable finding for your project report.

## Cell 8 — Save the trained model and its results

**Why:** we save the trained model so Step 4 (the Streamlit app) can load it directly without retraining, and we save the metrics to a small file so Step 3 (comparison against DistilBERT) can read them back programmatically instead of you having to copy numbers by hand.

In [8]:
import json

model.save_model("xgboost_phishing_model.json")

results = {
    "model": "XGBoost (classic ML, lexical features)",
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1
}
with open("xgboost_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved xgboost_phishing_model.json and xgboost_results.json")
print(results)

Saved xgboost_phishing_model.json and xgboost_results.json
{'model': 'XGBoost (classic ML, lexical features)', 'accuracy': 0.9927777777777778, 'precision': 0.9966405375139977, 'recall': 0.9888888888888889, 'f1': 0.992749581706637}


## Step 2 complete — what we have now

- 12 hand-engineered lexical/host-based features extracted from raw URL text
- A trained XGBoost model — this is your classic ML baseline
- Real, honest test-set metrics (accuracy, precision, recall, F1) and a confusion matrix
- Feature importances, giving you an explainable story about *why* the model works
- Saved model + results file, ready to be loaded later

**Next step (Step 3):** fine-tune DistilBERT directly on the raw URL text (no hand-engineered features needed — the transformer learns its own representations), then compare its test-set metrics against the XGBoost numbers saved here.